## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_export_api_key
load_dotenv(override=True)

# Use Claude as the model. Anthropic offers an OpenAI-compatible endpoint, so the Agents SDK can talk to it.
claude_client = AsyncOpenAI(base_url="https://api.anthropic.com/v1/", api_key=os.getenv("ANTHROPIC_API_KEY"))
claude = OpenAIChatCompletionsModel(model="claude-opus-5-5", openai_client=claude_client)

# Traces are uploaded to OpenAI's platform. The OpenAI key is used only for that upload; Claude still does the answering.
set_tracing_export_api_key(os.getenv("OPENAI_API_KEY"))


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model=claude)

In [3]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about GTA6")





In [4]:
# Here is the final output

print(result.final_output)

Why did the guy start training for GTA 6 by doing push-ups every day?

Because at the rate it's going, he'll need to be in shape to dodge the nursing home staff when he finally plays it.

**Bonus:** GTA 6 is so delayed that the in-game Vice City prices are already outdated.


In [5]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about GTA6', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Why did the guy start training for GTA 6 by doing push-ups every day?\n\nBecause at the rate it's going, he'll need to be in shape to dodge the nursing home staff when he finally plays it.\n\n**Bonus:** GTA 6 is so delayed that the in-game Vice City prices are already outdated.",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'claude-opus-5-5',
   'response_id': 'msg_011ChHyWkPQkxUq3CgSxuUTw'}}]

## Adding Observability with a trace

In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about  Germany")
print(result.final_output)

A German man walks into a bar and orders a martini.

The bartender asks, "Dry?"

The man replies, "Nein, just one!" 🍸

**Bonus:** Why don't Germans ever tell jokes at work?

Because humor isn't on the agenda until 15:30, and it's only allotted ten minutes. ⏰🇩🇪


## Now go and look at the trace

https://platform.openai.com/traces

In [ ]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are 5 jokes about AI agents:

1. **The Lightbulb**
How many AI agents does it take to change a lightbulb?
Five. One to change it, three to break the task into 47 subtasks, and one to write a 2,000-word report congratulating everyone on the successful illumination initiative.

2. **Full Autonomy**
My new AI agent is *fully autonomous*. It just needs me to click "Approve" before every single action it takes.

3. **The Inbox**
I told my AI agent to "get my inbox to zero."
It deleted my email account.
Technically, it's never been cleaner.

4. **The Bar Walk-In**
An AI agent walks into a bar.
Then it walks into the bar again.
And again.
The bartender says, "Are you stuck in a loop?"
The agent says, "Great question! Let me walk into the bar to find out."

5. **Success!**
My AI agent proudly reported: "Task completed successfully! ✅"
The task was: "Whatever you do, don't touch production."

## Part 2: Adding a tool

In [18]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [19]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [11]:
push("HEY!!")

Push: HEY!!


In [26]:
push

<function __main__.push(message)>

In [20]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [21]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x114953080>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [26]:
push_tool.description

'Send the given message to the user as a push notification'

In [27]:

notifier = Agent(name="Notifier", model=claude, instructions="You notify the user upon request", tools=[push_tool])

In [28]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


I sent the push notification saying "Your pizza is here! 🍕" and it went through successfully.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [29]:
agent = Agent(name="Assistant", model=claude)

In [30]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed, nice to meet you! What can I help you with today?


In [31]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't know your name. I don't have access to personal information about you, and you haven't mentioned it in our conversation. If you'd like, you can tell me, and I'll use it while we chat.


## Memory approach 1 - just manually pass in the list of dicts

In [32]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed, nice to meet you! What can I help you with today?


In [33]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hi Ed, nice to meet you! What can I help you with today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'claude-opus-5-5',
   'response_id': 'msg_011ChJ13MnamNtrd1CTHsWEk'}}]

In [34]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hi Ed, nice to meet you! What can I help you with today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'claude-opus-5-5',
   'response_id': 'msg_011ChJ13MnamNtrd1CTHsWEk'}},
 {'role': 'user', 'content': "What's my name?"}]

In [35]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed, as you told me at the start of our conversation.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [36]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("123_db")

In [37]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed, nice to meet you! What can I help you with today?


In [39]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed, as you told me at the start of our conversation.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>